In [62]:
import skimage.io
from skimage.measure import label, regionprops
from skimage.morphology import disk, binary_closing, binary_dilation
from scipy.ndimage import binary_fill_holes
import numpy as np
import numba as nb
# from numba.types import bool_
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
# from scipy import ndimage
from cupyx.scipy.ndimage import median_filter
import cupy as cp
import json
import glob, os, sys, math
import nd2
from numba import njit, jit, prange

import pathlib
from pathlib import WindowsPath, Path

In [2]:
def crop_center_square(data, side):
    """
    Extract a centered square of shape (t, side, side) from a 3D array (t, x, y).
    
    Parameters:
        data (np.ndarray): Input array with shape (t, x, y)
        side (int): Desired side length of square crop
    
    Returns:
        np.ndarray: Cropped array with shape (t, side, side)
    """
    t, x, y = data.shape
    assert side <= min(x, y), "Crop size must be smaller than both spatial dimensions."

    x_start = (x - side) // 2
    y_start = (y - side) // 2

    return data[:, x_start:x_start + side, y_start:y_start + side]

In [3]:
@jit(nopython=True)
def pad_with_reflect(data, pad_x, pad_y):
    """
    Reflective padding for a 3D array (t, x, y) along the spatial dimensions (x, y).

    Parameters:
        data (np.ndarray): Input array of shape (t, x, y).
        pad_x (int): Padding size along the x dimension.
        pad_y (int): Padding size along the y dimension.

    Returns:
        np.ndarray: Padded array.
    """
    t, x, y = data.shape
    padded = np.zeros((t, x + 2 * pad_x, y + 2 * pad_y), dtype=data.dtype)

    # Fill the central region
    padded[:, pad_x:pad_x + x, pad_y:pad_y + y] = data

    # Reflect padding on the edges
    # Top and bottom
    for px in range(pad_x):
        padded[:, px, pad_y:pad_y + y] = data[:, pad_x - px - 1, :]
        padded[:, x + pad_x + px, pad_y:pad_y + y] = data[:, x - px - 1, :]

    # Left and right
    for py in range(pad_y):
        padded[:, pad_x:pad_x + x, py] = padded[:, pad_x:pad_x + x, pad_y + pad_y - py - 1]
        padded[:, pad_x:pad_x + x, y + pad_y + py] = padded[:, pad_x:pad_x + x, y + pad_y - py - 1]

    # Corners (top-left, top-right, bottom-left, bottom-right)
    for px in range(pad_x):
        for py in range(pad_y):
            # Top-left
            padded[:, px, py] = data[:, pad_x - px - 1, pad_y - py - 1]
            # Top-right
            padded[:, px, y + pad_y + py] = data[:, pad_x - px - 1, y - py - 1]
            # Bottom-left
            padded[:, x + pad_x + px, py] = data[:, x - px - 1, pad_y - py - 1]
            # Bottom-right
            padded[:, x + pad_x + px, y + pad_y + py] = data[:, x - px - 1, y - py - 1]

    return padded

In [4]:
@jit(nopython=True, parallel=True)
def median_filter_1d_2d(img, filter_size):
    """
    Apply a median filter with filter size (1, H, W) to a 3D array using Numba.

    Parameters:
        img (np.ndarray): Input 3D array (shape: t, x, y).
        filter_size (tuple): Tuple of filter size (1, height, width).

    Returns:
        np.ndarray: Filtered 3D array.
    """
    data = np.copy(img)
    data = np.asarray(data, dtype=np.float32)

    # Extract the filter sizes
    f_t, f_x, f_y = filter_size

    if f_t != 1:
        raise ValueError("The filter size for the first dimension must be 1.")

    # Padding sizes
    pad_x = f_x // 2
    pad_y = f_y // 2

    # Pad the input data for spatial dimensions
    padded_data = pad_with_reflect(data, pad_x, pad_y)

    # Create an output array
    filtered_data = np.zeros_like(data)

    window_size = f_x * f_y
    k = window_size // 2  # Median index

    # Apply the filter frame by frame (parallel over time dimension)
    for t in prange(data.shape[0]):
        for x in range(data.shape[1]):
            for y in range(data.shape[2]):
                # Extract the spatial window for the current position
                window = padded_data[
                         t,  # Keep the time slice fixed
                         x:x + f_x,
                         y:y + f_y
                         ]
                # Compute the median and assign it to the output
                filtered_data[t, x, y] = np.median(window)
                # window = padded_data[t, x:x + f_x, y:y + f_y].flatten()
                # # Partial sort to find the median
                # median = np.partition(window, k)[k]
                # filtered_data[t, x, y] = median
    result_32bit = data - filtered_data

    # Step 3: Convert to 16-bit
    return result_32bit

In [5]:
@jit(nopython=True)
def random_sample_without_replacement(arr, sample_size):
    """
    Randomly sample `sample_size` elements from 1D array `arr` without replacement.
    Uses a Fisher-Yates shuffle on indices.
    """
    n = len(arr)
    indices = np.arange(n)
    for i in range(sample_size):
        j = np.random.randint(i, n)
        indices[i], indices[j] = indices[j], indices[i]
    sampled = np.empty(sample_size, dtype=arr.dtype)
    for i in range(sample_size):
        sampled[i] = arr[indices[i]]
    return sampled

@jit(nopython=True, parallel=True)
def median_filter_random_sampled(img, filter_size, sample_frac=0.05):
    """
    Median filter via random sampling, Numba-accelerated.

    Parameters:
        img (np.ndarray): Input (t, x, y), float32
        filter_size (tuple): Tuple of (1, fx, fy)
        sample_frac (float): Fraction of window to sample (e.g. 0.05 = 5%)

    Returns:
        np.ndarray: Residual image (input - background)
    """
    t, x, y = img.shape
    f_t, f_x, f_y = filter_size

    if f_t != 1:
        raise ValueError("Only spatial filtering supported (f_t must be 1).")

    pad_x = f_x // 2
    pad_y = f_y // 2
    padded = pad_with_reflect(img, pad_x, pad_y)
    filtered = np.zeros_like(img)

    window_size = f_x * f_y
    sample_size = max(1, int(window_size * sample_frac))

    for ti in prange(t):
        for xi in range(x):
            for yi in range(y):
                window = padded[ti, xi:xi + f_x, yi:yi + f_y].ravel()
                sampled = random_sample_without_replacement(window, sample_size)
                filtered[ti, xi, yi] = np.median(sampled)

    return img - filtered

In [98]:
# Reuse previous pad_with_reflect for 3D arrays (t, x, y) and modify to extract 2D padding

@njit
def reflect_pad_2d(img, pad):
    """
    Reflect-padding for a 2D image (Numba-compatible).
    """
    h, w = img.shape
    padded = np.zeros((h + 2 * pad, w + 2 * pad), dtype=img.dtype)

    # Center
    padded[pad:pad + h, pad:pad + w] = img

    # Top and bottom
    for i in range(pad):
        padded[i, pad:pad + w] = img[pad - i, :]
        padded[h + pad + i, pad:pad + w] = img[h - i - 2, :]

    # Left and right
    for j in range(pad):
        padded[:, j] = padded[:, 2 * pad - j]
        padded[:, w + pad + j] = padded[:, w + pad - j - 2]

    return padded

@njit
def find_median_from_hist(hist, total_count, min_val, max_val, num_bins):
    """
    Compute approximate median from histogram.
    """
    cum_sum = 0
    threshold = total_count // 2
    for b in range(num_bins):
        cum_sum += hist[b]
        if cum_sum >= threshold:
            return bin_center(b, min_val, max_val, num_bins)
    return 0.0  # fallback

@njit
def sliding_window_histogram_median_2d_reused_pad(image, window_size, min_val, max_val, num_bins):
    H, W = image.shape
    pad = window_size // 2
    padded = reflect_pad_2d(image, pad)
    result = np.zeros_like(image)

    for i in range(pad, H + pad):
        hist = np.zeros(num_bins, dtype=np.int32)

        # Initial histogram
        for dx in range(-pad, pad + 1):
            for dy in range(-pad, pad + 1):
                val = padded[i + dx, pad + dy]
                b = bin_index(val, min_val, max_val, num_bins)
                hist[b] += 1

        result[i - pad, 0] = find_median_from_hist(hist, window_size**2, min_val, max_val, num_bins)

        for j in range(1, W):
            # Remove left column
            for dx in range(-pad, pad + 1):
                val = padded[i + dx, j - 1]
                b = bin_index(val, min_val, max_val, num_bins)
                hist[b] -= 1
            # Add right column
            for dx in range(-pad, pad + 1):
                val = padded[i + dx, j + 2 * pad]
                b = bin_index(val, min_val, max_val, num_bins)
                hist[b] += 1

            result[i - pad, j] = find_median_from_hist(hist, window_size**2, min_val, max_val, num_bins)

    return result

@njit(parallel=True)
def histogram_median_filter_batch_time(data, window_size, min_val, max_val, num_bins):
    t, x, y = data.shape
    result = np.zeros_like(data)

    for ti in prange(t):
        result[ti] = sliding_window_histogram_median_2d_reused_pad(
            data[ti], window_size, min_val, max_val, num_bins
        )

    return data - result  # return residual (original - background)

In [6]:
import numpy as np
from numba import jit, prange

@jit(nopython=True)
def bin_index(val, min_val, max_val, num_bins):
    """
    Map a float value into a bin index.
    """
    if val <= min_val:
        return 0
    elif val >= max_val:
        return num_bins - 1
    return int((val - min_val) / (max_val - min_val) * (num_bins - 1))

@jit(nopython=True)
def bin_center(idx, min_val, max_val, num_bins):
    """
    Convert a bin index to the bin's center float value.
    """
    bin_width = (max_val - min_val) / num_bins
    return min_val + (idx + 0.5) * bin_width

@jit(nopython=True, parallel=True)
def median_filter_binned(img, filter_size, min_val=0.0, max_val=1.0, num_bins=256):
    """
    Approximate median filter using histogram binning, Numba-accelerated.

    Parameters:
        img (np.ndarray): Input (t, x, y), float32
        filter_size (tuple): Tuple of (1, fx, fy)
        min_val (float): Min pixel value for quantization
        max_val (float): Max pixel value for quantization
        num_bins (int): Number of bins for histogram

    Returns:
        np.ndarray: Residual image (input - background)
    """
    t, x, y = img.shape
    f_t, f_x, f_y = filter_size

    if f_t != 1:
        raise ValueError("Only spatial filtering supported (f_t must be 1).")

    pad_x = f_x // 2
    pad_y = f_y // 2
    padded = pad_with_reflect(img, pad_x, pad_y)
    filtered = np.zeros_like(img)

    total_pixels = f_x * f_y
    median_threshold = total_pixels // 2

    for ti in prange(t):
        for xi in range(x):
            for yi in range(y):
                # Initialize histogram
                hist = np.zeros(num_bins, dtype=np.int32)
                
                # Fill histogram
                for i in range(f_x):
                    for j in range(f_y):
                        val = padded[ti, xi + i, yi + j]
                        b = bin_index(val, min_val, max_val, num_bins)
                        hist[b] += 1

                # Find median bin
                cum_sum = 0
                for b in range(num_bins):
                    cum_sum += hist[b]
                    if cum_sum >= median_threshold:
                        filtered[ti, xi, yi] = bin_center(b, min_val, max_val, num_bins)
                        break

    return img - filtered


In [7]:
input_path = WindowsPath(r'H:\PROJECTS-03\Pablo\oscillating\pb-l15_comp\raw\pb-l15_comp_xy01.nd2')
channel = 1

In [8]:
with nd2.ND2File(input_path.as_posix()) as ndfile:
    f_channel = ndfile.asarray().astype(np.float32)[:,channel,:,:]

In [81]:
cropped = crop_center_square(f_channel, side=2048)

In [69]:
%%timeit -n1 -r3

filtered1 = median_filter_1d_2d(cropped[:3], filter_size=(1,141,141))

1min ± 746 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [25]:
%%time

filtered1 = median_filter_binned(cropped[:10], filter_size=(1,141,141))

CPU times: total: 55min 54s
Wall time: 5min 39s


In [26]:
skimage.io.imsave('./test_filtering_binned.tiff', filtered1)

In [27]:
%%time

filtered1 = median_filter_random_sampled(cropped[:10], filter_size=(1,141,141))

CPU times: total: 50min 43s
Wall time: 5min 6s


In [28]:
skimage.io.imsave('./test_filtering_random.tiff', filtered1)

In [54]:
single_frame = cropped[:,:,:]

In [23]:
%%time

# Convert to CuPy array
gpu_frame = cp.asarray(single_frame)

# Apply median filter
gpu_filtered = median_filter(gpu_frame, size=(1,141, 141))

filtered_np = gpu_frame - gpu_filtered

# Convert back to NumPy if needed
filtered_np = cp.asnumpy(gpu_filtered)

CPU times: total: 5min 42s
Wall time: 5min 42s


In [24]:
%%time
result = median_filter_random_sampled(single_frame, filter_size=(1,141,141))

In [73]:
cropped[0].shape[1]

512

In [101]:
%%time

filtered = histogram_median_filter_batch_time(
    cropped[:,:,:], window_size=161, min_val=0.0, max_val=65535.0, num_bins=8912)


CPU times: total: 29min 17s
Wall time: 53.5 s


In [102]:
skimage.io.imsave('./sliding_window_reuse.tiff', filtered)

In [59]:
%%time

from cupyx.scipy.ndimage import grey_erosion
from cupyx.scipy.ndimage import grey_dilation
from cupyx.scipy.ndimage import grey_opening


gpu_frame = cp.asarray(single_frame)
gpu_morph = grey_opening(gpu_frame, size=(1,401,401))


filtered_np = gpu_frame - gpu_morph 

filtered_np = cp.asnumpy(filtered_np)

CPU times: total: 15.3 s
Wall time: 15.3 s


In [60]:
skimage.io.imsave('./test_filtering_grey_open.tiff', filtered_np)

In [14]:
%%time

scipy.ndimage.median_filter(single_frame, size=(1,141,141))

CPU times: total: 2min 49s
Wall time: 2min 49s


array([[[1452., 1451., 1451., ..., 1445., 1446., 1446.],
        [1452., 1451., 1451., ..., 1446., 1446., 1446.],
        [1452., 1451., 1451., ..., 1445., 1446., 1446.],
        ...,
        [1477., 1477., 1476., ..., 1481., 1481., 1481.],
        [1477., 1477., 1476., ..., 1481., 1481., 1481.],
        [1477., 1477., 1476., ..., 1481., 1481., 1481.]],

       [[1446., 1447., 1447., ..., 1447., 1447., 1447.],
        [1447., 1447., 1447., ..., 1447., 1447., 1447.],
        [1447., 1447., 1447., ..., 1447., 1447., 1447.],
        ...,
        [1480., 1480., 1480., ..., 1487., 1487., 1487.],
        [1480., 1480., 1480., ..., 1487., 1487., 1487.],
        [1480., 1480., 1480., ..., 1487., 1487., 1487.]],

       [[1441., 1441., 1441., ..., 1438., 1438., 1438.],
        [1441., 1441., 1441., ..., 1438., 1438., 1438.],
        [1441., 1441., 1441., ..., 1438., 1438., 1438.],
        ...,
        [1467., 1467., 1466., ..., 1478., 1477., 1477.],
        [1466., 1466., 1466., ..., 1478., 147

In [36]:
import cupy

print("CuPy version:", cupy.__version__)
print("CUDA available:", cupy.is_available())
print("CUDA runtime version:", cupy.cuda.runtime.getVersion())
print("CUDA driver version:", cupy.cuda.driver.get_version())
print("GPU name:", cupy.cuda.runtime.getDeviceProperties(0)['name'])


CuPy version: 11.6.0
CUDA available: True


AttributeError: module 'cupy_backends.cuda.api.runtime' has no attribute 'getVersion'